# USAA Complaints from CFPB API

This script pulls all USAA complaints from the Consumer Financial Protection Bureau (CFPB) complaint database API.

In [1]:
import requests
import pandas as pd
import json
from datetime import datetime
import time

In [2]:
cmp_data= pd.read_csv('complaints.csv')  # Assuming a CSV file with company data
cmp_data

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2020-07-06,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,346XX,NaN,Other,Web,2020-07-06,Closed with explanation,Yes,NaN,3730948
1,2025-10-14,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information is missing that should be on the r...,NaN,NaN,"EQUIFAX, INC.",TX,75062,NaN,NaN,Web,2025-10-14,In progress,Yes,NaN,16558024
2,2025-10-10,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"EQUIFAX, INC.",GA,30341,NaN,NaN,Web,2025-10-10,In progress,Yes,NaN,16507707
3,2025-10-15,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,75287,NaN,NaN,Web,2025-10-15,In progress,Yes,NaN,16593757
4,2025-10-16,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,Experian Information Solutions Inc.,NC,28379,NaN,NaN,Web,2025-10-16,In progress,Yes,NaN,16623506
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11522170,2014-04-21,Credit card,NaN,Other,NaN,NaN,NaN,AMERICAN EXPRESS COMPANY,CA,92003,NaN,NaN,Web,2014-04-21,Closed with monetary relief,Yes,No,816552
11522171,2019-07-26,Mortgage,Conventional home mortgage,Struggling to pay mortgage,NaN,NaN,NaN,JPMORGAN CHASE & CO.,NaN,NaN,NaN,NaN,Referral,2019-07-29,Closed with explanation,Yes,NaN,3322081
11522172,2019-05-22,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",NaN,NaN,NaN,Consent not provided,Web,2019-05-22,Closed with explanation,Yes,NaN,3249858
11522173,2019-06-03,"Credit reporting, credit repair services, or o...",Credit reporting,Problem with a credit reporting company's inve...,Their investigation did not fix an error on yo...,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",NaN,NaN,NaN,Consent not provided,Web,2019-06-03,Closed with non-monetary relief,Yes,NaN,3262839


In [8]:
#count per company
company_counts = cmp_data['Company'].value_counts().reset_index()
company_counts.columns = ['Company', 'Count']   
company_counts.to_csv('company_counts.csv', index=False)

In [5]:
def fetch_complaints(api_url, params, max_records=50):
    all_complaints = []
    total_fetched = 0
    page = 1
    records_per_request = params.get('size', 10)

    while total_fetched < max_records:
        params['page'] = page
        response = requests.get(api_url, params=params)
        data = response.text
        #print(data)
        #write to file for debugging
        with open('debug_response.txt', 'w') as f:
            f.write(data)
        
        
        # data = json.loads(data)

        
        # complaints = data.get('results', [])
        # if not complaints:
        #     break
        
        # all_complaints.extend(complaints)
        # total_fetched += len(complaints)
        # page += 1
        
        # # To avoid hitting rate limits
        # time.sleep(1)

    return all_complaints[:max_records]
# Example usage
api_url = 'https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/'
params = {
    'size': 10,
    'sort': 'created_date_desc',
    'format': 'json'
}
complaints = fetch_complaints(api_url, params, max_records=50)

ChunkedEncodingError: ("Connection broken: ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)", ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

In [ ]:
def fetch_usaa_complaints():
    """
    Fetch all USAA complaints from the CFPB complaint database API
    """
    base_url = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
    
    # Parameters to filter for USAA complaints
    params = {
       ### 'company': 'USAA',  # Filter for USAA company
        'size': 10,  # Maximum records per request (API limit)
        'sort': 'created_date_desc',  # Sort by newest first
        'format': 'json'
    }
    
    all_complaints = []
    offset = 0
    
    print("Fetching USAA complaints from CFPB API...")
    
    while True:
        # Add offset for pagination
        params['from'] = offset
        
        try:
            # Make API request
            response = requests.get(base_url, params=params, timeout=30)
            response.raise_for_status()
            
            data = response.json()
            
            # Extract complaints from response
            if 'hits' in data and 'hits' in data['hits']:
                complaints = data['hits']['hits']
                
                if not complaints:  # No more complaints to fetch
                    break
                
                # Add complaints to our list
                for complaint in complaints:
                    complaint_data = complaint['_source']
                    all_complaints.append(complaint_data)
                
                print(f"Fetched {len(complaints)} complaints (Total: {len(all_complaints)})")
                
                # Update offset for next batch
                offset += len(complaints)
                
                # Add small delay to be respectful to the API
                time.sleep(0.5)
                
            else:
                print("No more complaints found or unexpected response format")
                break
                
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            break
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON response: {e}")
            break
    
    print(f"Total USAA complaints fetched: {len(all_complaints)}")
    return all_complaints

In [ ]:
def process_complaints_data(complaints):
    """
    Convert complaints list to pandas DataFrame and perform basic processing
    """
    if not complaints:
        print("No complaints data to process")
        return pd.DataFrame()
    
    # Create DataFrame
    df = pd.DataFrame(complaints)
    
    # Convert date columns to datetime
    date_columns = ['date_received', 'date_sent_to_company']
    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    
    # Display basic info
    print(f"\nDataset Info:")
    print(f"Total complaints: {len(df)}")
    print(f"Date range: {df['date_received'].min()} to {df['date_received'].max()}")
    print(f"Columns: {list(df.columns)}")
    
    return df

def analyze_complaints(df):
    """
    Perform basic analysis on the complaints data
    """
    if df.empty:
        return
    
    print("\n=== USAA Complaints Analysis ===")
    
    # Top products with complaints
    if 'product' in df.columns:
        print("\nTop 10 Products by Complaint Count:")
        print(df['product'].value_counts().head(10))
    
    # Top issues
    if 'issue' in df.columns:
        print("\nTop 10 Issues:")
        print(df['issue'].value_counts().head(10))
    
    # Complaints by year
    if 'date_received' in df.columns:
        print("\nComplaints by Year:")
        df['year'] = df['date_received'].dt.year
        print(df['year'].value_counts().sort_index())
    
    # Company response
    if 'company_response_to_consumer' in df.columns:
        print("\nCompany Response Types:")
        print(df['company_response_to_consumer'].value_counts())
    
    # Consumer disputed
    if 'consumer_disputed' in df.columns:
        print("\nConsumer Disputed Response:")
        print(df['consumer_disputed'].value_counts())
    
    return df

In [ ]:
# Execute the script
if __name__ == "__main__":
    # Fetch all USAA complaints
    complaints_data = fetch_usaa_complaints()
    print(f"Total complaints fetched: {len(complaints_data)}")
    complaints_data
    # Process the data into a DataFrame
    #usaa_complaints_df = process_complaints_data(complaints_data)
    
    # Perform analysis
    # if not usaa_complaints_df.empty:
    #     analyzed_df = analyze_complaints(usaa_complaints_df)
        
    #     # Save to CSV for further analysis
    #     timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    #     filename = f"usaa_complaints_{timestamp}.csv"
    #     usaa_complaints_df.to_csv(filename, index=False)
    #     print(f"\nData saved to: {filename}")
        
    #     # Display first few rows
    #     print("\nFirst 5 complaints:")
    #     print(usaa_complaints_df.head())
    # else:
    #     print("No complaints data retrieved")

In [ ]:
# Optional: Create visualizations (requires matplotlib and seaborn)
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    def create_visualizations(df):
        """
        Create basic visualizations of the complaints data
        """
        if df.empty:
            return
        
        plt.style.use('default')
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle('USAA Complaints Analysis', fontsize=16, fontweight='bold')
        
        # 1. Complaints by Product
        if 'product' in df.columns:
            top_products = df['product'].value_counts().head(10)
            sns.barplot(x=top_products.values, y=top_products.index, ax=axes[0,0])
            axes[0,0].set_title('Top 10 Products by Complaint Count')
            axes[0,0].set_xlabel('Number of Complaints')
        
        # 2. Complaints over time
        if 'date_received' in df.columns:
            df_monthly = df.set_index('date_received').resample('M').size()
            axes[0,1].plot(df_monthly.index, df_monthly.values)
            axes[0,1].set_title('Complaints Over Time (Monthly)')
            axes[0,1].set_xlabel('Date')
            axes[0,1].set_ylabel('Number of Complaints')
            axes[0,1].tick_params(axis='x', rotation=45)
        
        # 3. Company Response Distribution
        if 'company_response_to_consumer' in df.columns:
            response_counts = df['company_response_to_consumer'].value_counts()
            axes[1,0].pie(response_counts.values, labels=response_counts.index, autopct='%1.1f%%')
            axes[1,0].set_title('Company Response Distribution')
        
        # 4. Top Issues
        if 'issue' in df.columns:
            top_issues = df['issue'].value_counts().head(8)
            sns.barplot(x=top_issues.values, y=top_issues.index, ax=axes[1,1])
            axes[1,1].set_title('Top 8 Issues')
            axes[1,1].set_xlabel('Number of Complaints')
        
        plt.tight_layout()
        plt.show()
    
    # Create visualizations if data exists
    if 'usaa_complaints_df' in locals() and not usaa_complaints_df.empty:
        create_visualizations(usaa_complaints_df)
    
except ImportError:
    print("Matplotlib and/or Seaborn not installed. Skipping visualizations.")
    print("Install with: pip install matplotlib seaborn")

In [ ]:
# Additional imports for Hugging Face and clustering
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Hugging Face imports
try:
    from transformers import AutoTokenizer, AutoModel
    import torch
    from huggingface_hub import login, HfApi
    print("Hugging Face libraries loaded successfully")
except ImportError as e:
    print(f"Please install required packages: pip install transformers torch huggingface_hub")
    print(f"Error: {e}")

In [ ]:
# Hugging Face Hub Connection
def connect_to_huggingface():
    """
    Connect to Hugging Face Hub
    Note: You may need to login with your HF token for some models
    """
    try:
        # Optional: Login if you have a token (uncomment and add your token)
        # login(token="your_hf_token_here")
        
        api = HfApi()
        print("Successfully connected to Hugging Face Hub")
        return api
    except Exception as e:
        print(f"Error connecting to Hugging Face Hub: {e}")
        print("You may need to set up your HF token for private models")
        return None

# Initialize HF connection
hf_api = connect_to_huggingface()

In [ ]:
class GemmaEmbedder:
    """
    Wrapper class for Google's Gemma model embeddings
    """
    def __init__(self, model_name="google/gemma-2b"):
        """
        Initialize Gemma embedder
        Note: Using gemma-2b as it's more accessible. For embeddings, we'll use the hidden states.
        """
        self.model_name = model_name
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")
        
        try:
            print(f"Loading {model_name}...")
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModel.from_pretrained(
                model_name,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map="auto" if torch.cuda.is_available() else None
            )
            
            # Add padding token if not present
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
                
            print("Gemma model loaded successfully!")
            
        except Exception as e:
            print(f"Error loading Gemma model: {e}")
            print("Trying alternative embedding model...")
            # Fallback to a sentence transformer model
            try:
                from sentence_transformers import SentenceTransformer
                self.model = SentenceTransformer('all-MiniLM-L6-v2')
                self.is_sentence_transformer = True
                print("Using SentenceTransformer as fallback")
            except ImportError:
                print("Please install sentence-transformers: pip install sentence-transformers")
                raise
    
    def encode_texts(self, texts, max_length=512, batch_size=8):
        """
        Encode a list of texts into embeddings
        """
        if hasattr(self, 'is_sentence_transformer'):
            # Use sentence transformer
            return self.model.encode(texts, show_progress_bar=True)
        
        embeddings = []
        
        # Process in batches
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            
            # Tokenize
            inputs = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(self.device)
            
            # Get embeddings
            with torch.no_grad():
                outputs = self.model(**inputs)
                # Use mean pooling of last hidden states
                embeddings_batch = outputs.last_hidden_state.mean(dim=1)
                embeddings.extend(embeddings_batch.cpu().numpy())
            
            if i % (batch_size * 10) == 0:
                print(f"Processed {i + len(batch_texts)}/{len(texts)} texts")
        
        return np.array(embeddings)

# Initialize Gemma embedder
print("Initializing Gemma embedder...")
gemma_embedder = GemmaEmbedder()

In [ ]:
def prepare_complaint_texts(df, max_samples=1000):
    """
    Prepare complaint texts for embedding
    Combines relevant text fields and limits sample size for computational efficiency
    """
    if df.empty:
        print("No data to process")
        return [], df
    
    # Combine relevant text fields
    text_fields = ['consumer_complaint_narrative', 'issue', 'sub_issue', 'product', 'sub_product']
    available_fields = [field for field in text_fields if field in df.columns]
    
    print(f"Available text fields: {available_fields}")
    
    # Create combined text
    complaint_texts = []
    processed_df = df.copy()
    
    for idx, row in df.iterrows():
        text_parts = []
        
        for field in available_fields:
            if pd.notna(row[field]) and str(row[field]).strip():
                text_parts.append(f"{field}: {str(row[field]).strip()}")
        
        combined_text = " | ".join(text_parts)
        complaint_texts.append(combined_text if combined_text else "No text available")
    
    # Limit sample size if too large
    if len(complaint_texts) > max_samples:
        print(f"Limiting to {max_samples} samples for computational efficiency")
        indices = np.random.choice(len(complaint_texts), max_samples, replace=False)
        complaint_texts = [complaint_texts[i] for i in indices]
        processed_df = processed_df.iloc[indices].reset_index(drop=True)
    
    print(f"Prepared {len(complaint_texts)} complaint texts for embedding")
    return complaint_texts, processed_df

def generate_embeddings(texts, embedder):
    """
    Generate embeddings for complaint texts
    """
    print("Generating embeddings...")
    embeddings = embedder.encode_texts(texts)
    print(f"Generated embeddings shape: {embeddings.shape}")
    return embeddings

In [ ]:
def perform_agglomerative_clustering(embeddings, distance_thresholds=None, linkage='ward'):
    """
    Perform agglomerative clustering with different distance thresholds
    """
    if distance_thresholds is None:
        distance_thresholds = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
    
    clustering_results = {}
    
    print(f"Performing agglomerative clustering with linkage='{linkage}'")
    print(f"Distance thresholds: {distance_thresholds}")
    
    for threshold in distance_thresholds:
        print(f"\nClustering with distance threshold: {threshold}")
        
        # Perform clustering
        clustering = AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=threshold,
            linkage=linkage
        )
        
        cluster_labels = clustering.fit_predict(embeddings)
        n_clusters = len(np.unique(cluster_labels))
        
        # Calculate silhouette score (if more than 1 cluster)
        silhouette = None
        if n_clusters > 1 and n_clusters < len(embeddings):
            try:
                silhouette = silhouette_score(embeddings, cluster_labels)
            except:
                silhouette = None
        
        clustering_results[threshold] = {
            'labels': cluster_labels,
            'n_clusters': n_clusters,
            'silhouette_score': silhouette,
            'cluster_sizes': np.bincount(cluster_labels)
        }
        
        print(f"  Number of clusters: {n_clusters}")
        print(f"  Silhouette score: {silhouette:.3f}" if silhouette else "  Silhouette score: N/A")
        print(f"  Cluster sizes: {dict(enumerate(np.bincount(cluster_labels)))}")
    
    return clustering_results

def analyze_clusters(clustering_results, texts, df, top_n=5):
    """
    Analyze and display cluster characteristics
    """
    print("\n" + "="*80)
    print("CLUSTER ANALYSIS RESULTS")
    print("="*80)
    
    for threshold, results in clustering_results.items():
        print(f"\n{'='*20} Distance Threshold: {threshold} {'='*20}")
        print(f"Number of clusters: {results['n_clusters']}")
        print(f"Silhouette score: {results['silhouette_score']:.3f}" if results['silhouette_score'] else "Silhouette score: N/A")
        
        labels = results['labels']
        unique_labels = np.unique(labels)
        
        # Show sample texts from each cluster
        for cluster_id in unique_labels[:top_n]:  # Show only top N clusters
            cluster_mask = labels == cluster_id
            cluster_texts = [texts[i] for i in range(len(texts)) if cluster_mask[i]]
            cluster_size = sum(cluster_mask)
            
            print(f"\n--- Cluster {cluster_id} (Size: {cluster_size}) ---")
            
            # Show first few texts from this cluster
            for i, text in enumerate(cluster_texts[:3]):
                print(f"  {i+1}. {text[:200]}...")
            
            if len(cluster_texts) > 3:
                print(f"  ... and {len(cluster_texts) - 3} more")
    
    return clustering_results

In [ ]:
def visualize_clustering_results(embeddings, clustering_results, max_plots=4):
    """
    Visualize clustering results using PCA for dimensionality reduction
    """
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        
        # Reduce dimensionality for visualization
        pca = PCA(n_components=2)
        embeddings_2d = pca.fit_transform(embeddings)
        
        # Select best thresholds to visualize
        thresholds_to_plot = list(clustering_results.keys())[:max_plots]
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        axes = axes.flatten()
        
        for i, threshold in enumerate(thresholds_to_plot):
            if i >= len(axes):
                break
                
            results = clustering_results[threshold]
            labels = results['labels']
            n_clusters = results['n_clusters']
            silhouette = results['silhouette_score']
            
            # Create scatter plot
            scatter = axes[i].scatter(
                embeddings_2d[:, 0], 
                embeddings_2d[:, 1], 
                c=labels, 
                cmap='tab20', 
                alpha=0.7,
                s=50
            )
            
            axes[i].set_title(f'Threshold: {threshold} | Clusters: {n_clusters}\nSilhouette: {silhouette:.3f}' if silhouette else f'Threshold: {threshold} | Clusters: {n_clusters}')
            axes[i].set_xlabel('PCA Component 1')
            axes[i].set_ylabel('PCA Component 2')
            
            # Add colorbar
            plt.colorbar(scatter, ax=axes[i])
        
        # Hide unused subplots
        for i in range(len(thresholds_to_plot), len(axes)):
            axes[i].axis('off')
        
        plt.tight_layout()
        plt.suptitle('Agglomerative Clustering Results (PCA Visualization)', fontsize=16, y=1.02)
        plt.show()
        
        # Plot clustering metrics
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        thresholds = list(clustering_results.keys())
        n_clusters_list = [clustering_results[t]['n_clusters'] for t in thresholds]
        silhouette_scores = [clustering_results[t]['silhouette_score'] for t in thresholds]
        
        # Number of clusters vs threshold
        ax1.plot(thresholds, n_clusters_list, 'bo-')
        ax1.set_xlabel('Distance Threshold')
        ax1.set_ylabel('Number of Clusters')
        ax1.set_title('Number of Clusters vs Distance Threshold')
        ax1.grid(True)
        
        # Silhouette score vs threshold
        valid_silhouettes = [(t, s) for t, s in zip(thresholds, silhouette_scores) if s is not None]
        if valid_silhouettes:
            valid_thresholds, valid_scores = zip(*valid_silhouettes)
            ax2.plot(valid_thresholds, valid_scores, 'ro-')
            ax2.set_xlabel('Distance Threshold')
            ax2.set_ylabel('Silhouette Score')
            ax2.set_title('Silhouette Score vs Distance Threshold')
            ax2.grid(True)
        else:
            ax2.text(0.5, 0.5, 'No valid silhouette scores', ha='center', va='center', transform=ax2.transAxes)
        
        plt.tight_layout()
        plt.show()
        
    except ImportError:
        print("Matplotlib/Seaborn not available for visualization")
    except Exception as e:
        print(f"Error creating visualizations: {e}")

def find_optimal_threshold(clustering_results):
    """
    Find optimal distance threshold based on silhouette score
    """
    valid_results = {
        t: r for t, r in clustering_results.items() 
        if r['silhouette_score'] is not None and r['n_clusters'] > 1
    }
    
    if not valid_results:
        print("No valid clustering results found")
        return None
    
    # Find threshold with highest silhouette score
    best_threshold = max(valid_results.keys(), key=lambda t: valid_results[t]['silhouette_score'])
    best_score = valid_results[best_threshold]['silhouette_score']
    best_n_clusters = valid_results[best_threshold]['n_clusters']
    
    print(f"\nOptimal clustering:")
    print(f"  Distance threshold: {best_threshold}")
    print(f"  Number of clusters: {best_n_clusters}")
    print(f"  Silhouette score: {best_score:.3f}")
    
    return best_threshold, valid_results[best_threshold]

In [ ]:
# Main execution for clustering pipeline
def run_complaint_clustering_pipeline(df, max_samples=1000):
    """
    Run the complete clustering pipeline on USAA complaints
    """
    print("="*80)
    print("USAA COMPLAINTS CLUSTERING PIPELINE")
    print("="*80)
    
    # Step 1: Prepare texts
    print("\n1. Preparing complaint texts...")
    complaint_texts, processed_df = prepare_complaint_texts(df, max_samples=max_samples)
    
    if not complaint_texts:
        print("No complaint texts to process")
        return None
    
    # Step 2: Generate embeddings
    print("\n2. Generating embeddings with Gemma...")
    embeddings = generate_embeddings(complaint_texts, gemma_embedder)
    
    # Step 3: Perform clustering with different thresholds
    print("\n3. Performing agglomerative clustering...")
    distance_thresholds = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
    clustering_results = perform_agglomerative_clustering(
        embeddings, 
        distance_thresholds=distance_thresholds
    )
    
    # Step 4: Analyze clusters
    print("\n4. Analyzing clusters...")
    analyze_clusters(clustering_results, complaint_texts, processed_df)
    
    # Step 5: Find optimal threshold
    print("\n5. Finding optimal clustering...")
    optimal_result = find_optimal_threshold(clustering_results)
    
    # Step 6: Visualize results
    print("\n6. Creating visualizations...")
    visualize_clustering_results(embeddings, clustering_results)
    
    # Step 7: Save results
    print("\n7. Saving clustering results...")
    if optimal_result:
        best_threshold, best_results = optimal_result
        processed_df['cluster_label'] = best_results['labels']
        
        # Save clustered data
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        clustered_filename = f"usaa_complaints_clustered_{timestamp}.csv"
        processed_df.to_csv(clustered_filename, index=False)
        print(f"Clustered data saved to: {clustered_filename}")
        
        # Save cluster summary
        cluster_summary = []
        for cluster_id in np.unique(best_results['labels']):
            cluster_mask = best_results['labels'] == cluster_id
            cluster_texts_sample = [complaint_texts[i] for i in range(len(complaint_texts)) if cluster_mask[i]][:5]
            
            cluster_summary.append({
                'cluster_id': cluster_id,
                'size': sum(cluster_mask),
                'sample_texts': cluster_texts_sample
            })
        
        summary_filename = f"cluster_summary_{timestamp}.json"
        import json
        with open(summary_filename, 'w') as f:
            json.dump(cluster_summary, f, indent=2)
        print(f"Cluster summary saved to: {summary_filename}")
    
    return {
        'embeddings': embeddings,
        'clustering_results': clustering_results,
        'texts': complaint_texts,
        'processed_df': processed_df
    }

# Execute clustering pipeline if we have complaint data
if 'usaa_complaints_df' in locals() and not usaa_complaints_df.empty:
    print("Running clustering pipeline on USAA complaints...")
    clustering_pipeline_results = run_complaint_clustering_pipeline(usaa_complaints_df, max_samples=500)
else:
    print("No USAA complaints data found. Please run the data fetching cells first.")
    print("You can also test with a small sample:")
    
    # Create sample data for testing
    sample_complaints = pd.DataFrame({
        'consumer_complaint_narrative': [
            'I was charged overdraft fees unfairly',
            'Credit card billing error not resolved',
            'Mortgage payment processing issues',
            'Insurance claim denied wrongfully',
            'ATM fees are too high',
            'Credit report has incorrect information',
            'Loan application was rejected unfairly',
            'Customer service was unhelpful'
        ],
        'issue': [
            'Overdraft fees', 'Billing error', 'Payment processing', 
            'Claim denial', 'ATM fees', 'Credit report', 
            'Loan rejection', 'Customer service'
        ],
        'product': [
            'Checking account', 'Credit card', 'Mortgage', 
            'Insurance', 'Checking account', 'Credit reporting',
            'Personal loan', 'General'
        ]
    })
    
    print("Running clustering pipeline on sample data...")
    clustering_pipeline_results = run_complaint_clustering_pipeline(sample_complaints, max_samples=100)

In [ ]:
class ClusterNamer:
    """
    Class to generate cluster names using LLM hosted on Hugging Face
    """
    def __init__(self, model_name="microsoft/DialoGPT-medium", hf_token=None):
        """
        Initialize cluster namer with HuggingFace model
        
        Popular models for text generation:
        - "microsoft/DialoGPT-medium"
        - "google/flan-t5-base" 
        - "mistralai/Mistral-7B-Instruct-v0.1"
        - "meta-llama/Llama-2-7b-chat-hf"
        """
        self.model_name = model_name
        self.hf_token = hf_token
        
        try:
            from huggingface_hub import InferenceClient
            self.client = InferenceClient(token=hf_token)
            print(f"Initialized ClusterNamer with model: {model_name}")
        except ImportError:
            print("Please install huggingface_hub: pip install huggingface_hub")
            raise
        except Exception as e:
            print(f"Error initializing InferenceClient: {e}")
            # Fallback to direct API calls
            self.client = None
    
    def generate_cluster_name(self, sample_texts, max_samples=5):
        """
        Generate a short descriptive name for a cluster based on sample texts
        """
        # Limit and clean sample texts
        clean_samples = []
        for text in sample_texts[:max_samples]:
            # Extract just the narrative part if available
            if "consumer_complaint_narrative:" in text:
                narrative = text.split("consumer_complaint_narrative:")[1].split("|")[0].strip()
                clean_samples.append(narrative[:200])  # Limit length
            else:
                clean_samples.append(text[:200])
        
        # Create prompt for naming
        samples_text = "\n".join([f"- {text}" for text in clean_samples])
        
        prompt = f"""Based on these customer complaint examples, provide a short 2-4 word descriptive name for this cluster of complaints:

{samples_text}

Cluster name:"""
        
        try:
            if self.client:
                # Use InferenceClient
                response = self.client.text_generation(
                    prompt,
                    model=self.model_name,
                    max_new_tokens=20,
                    temperature=0.3,
                    do_sample=True,
                    stop_sequences=["\n", ".", "Cluster"]
                )
                
                if isinstance(response, str):
                    cluster_name = response.strip()
                else:
                    cluster_name = str(response).strip()
                    
            else:
                # Fallback to direct API call
                cluster_name = self._api_call_fallback(prompt)
            
            # Clean up the response
            cluster_name = cluster_name.replace("Cluster name:", "").strip()
            cluster_name = cluster_name.split("\n")[0].strip()
            
            # Limit to reasonable length
            if len(cluster_name) > 50:
                cluster_name = cluster_name[:50] + "..."
            
            return cluster_name if cluster_name else "Unnamed Cluster"
            
        except Exception as e:
            print(f"Error generating cluster name: {e}")
            # Fallback to keyword-based naming
            return self._fallback_naming(clean_samples)
    
    def _api_call_fallback(self, prompt):
        """
        Fallback to direct API calls if InferenceClient fails
        """
        import requests
        
        headers = {
            "Authorization": f"Bearer {self.hf_token}",
            "Content-Type": "application/json"
        }
        
        payload = {
            "inputs": prompt,
            "parameters": {
                "max_new_tokens": 20,
                "temperature": 0.3,
                "do_sample": True,
                "stop": ["\n", ".", "Cluster"]
            }
        }
        
        response = requests.post(
            f"https://api-inference.huggingface.co/models/{self.model_name}",
            headers=headers,
            json=payload,
            timeout=30
        )
        
        if response.status_code == 200:
            result = response.json()
            if isinstance(result, list) and len(result) > 0:
                return result[0].get('generated_text', '').replace(prompt, '').strip()
        
        return ""
    
    def _fallback_naming(self, texts):
        """
        Simple keyword-based fallback naming
        """
        # Extract common keywords
        keywords = []
        for text in texts:
            words = text.lower().split()
            # Look for financial terms
            financial_terms = ['fee', 'charge', 'payment', 'account', 'credit', 'debit', 
                             'loan', 'mortgage', 'insurance', 'overdraft', 'billing']
            for term in financial_terms:
                if term in words:
                    keywords.append(term)
        
        if keywords:
            # Get most common keyword
            from collections import Counter
            most_common = Counter(keywords).most_common(1)[0][0]
            return f"{most_common.title()} Issues"
        
        return "General Complaints"

def name_all_clusters(clustering_results, complaint_texts, threshold, sample_size=5, hf_token=None):
    """
    Generate names for all clusters using LLM
    """
    print(f"\nGenerating cluster names for threshold {threshold}...")
    
    # Initialize cluster namer
    try:
        # Try with a good instruction-following model
        cluster_namer = ClusterNamer("google/flan-t5-base", hf_token=hf_token)
    except:
        try:
            # Fallback model
            cluster_namer = ClusterNamer("microsoft/DialoGPT-medium", hf_token=hf_token)
        except:
            print("Could not initialize cluster namer. Using fallback naming.")
            cluster_namer = None
    
    results = clustering_results[threshold]
    labels = results['labels']
    unique_labels = np.unique(labels)
    
    cluster_names = {}
    
    for cluster_id in unique_labels:
        cluster_mask = labels == cluster_id
        cluster_texts = [complaint_texts[i] for i in range(len(complaint_texts)) if cluster_mask[i]]
        cluster_size = len(cluster_texts)
        
        print(f"  Naming cluster {cluster_id} (size: {cluster_size})...")
        
        if cluster_namer:
            # Sample texts for naming
            sample_texts = cluster_texts[:sample_size]
            cluster_name = cluster_namer.generate_cluster_name(sample_texts)
        else:
            # Fallback naming
            cluster_name = f"Cluster {cluster_id}"
        
        cluster_names[cluster_id] = {
            'name': cluster_name,
            'size': cluster_size,
            'sample_texts': cluster_texts[:3]  # Keep few samples for reference
        }
        
        print(f"    → '{cluster_name}'")
    
    return cluster_names

def display_named_clusters(cluster_names, threshold):
    """
    Display clusters with their generated names
    """
    print(f"\n{'='*60}")
    print(f"NAMED CLUSTERS (Distance Threshold: {threshold})")
    print(f"{'='*60}")
    
    for cluster_id, info in cluster_names.items():
        print(f"\n🏷️  Cluster {cluster_id}: '{info['name']}' (Size: {info['size']})")
        print("   Sample complaints:")
        for i, text in enumerate(info['sample_texts'], 1):
            # Clean up text for display
            display_text = text.replace("consumer_complaint_narrative:", "").split("|")[0].strip()
            print(f"   {i}. {display_text[:150]}...")

# Initialize cluster namer (you may need to provide HF token for some models)
print("Setting up cluster naming...")
HF_TOKEN = None  # Set your HuggingFace token here if needed for private models

def enhanced_clustering_with_naming(df, max_samples=500, distance_thresholds=None):
    """
    Enhanced clustering pipeline that includes LLM-based cluster naming
    """
    print("="*80)
    print("ENHANCED USAA COMPLAINTS CLUSTERING WITH LLM NAMING")
    print("="*80)
    
    # Run the standard clustering pipeline
    if distance_thresholds is None:
        distance_thresholds = [1.0, 1.5, 2.0, 2.5, 3.0]
    
    # Step 1-6: Run standard pipeline
    print("\n1-6. Running standard clustering pipeline...")
    
    # Prepare texts
    complaint_texts, processed_df = prepare_complaint_texts(df, max_samples=max_samples)
    if not complaint_texts:
        return None
    
    # Generate embeddings
    embeddings = generate_embeddings(complaint_texts, gemma_embedder)
    
    # Perform clustering
    clustering_results = perform_agglomerative_clustering(
        embeddings, 
        distance_thresholds=distance_thresholds
    )
    
    # Find optimal threshold
    optimal_result = find_optimal_threshold(clustering_results)
    
    # Step 7: Generate cluster names using LLM
    print("\n7. Generating cluster names using LLM...")
    all_cluster_names = {}
    
    if optimal_result:
        best_threshold, _ = optimal_result
        cluster_names = name_all_clusters(
            clustering_results, 
            complaint_texts, 
            best_threshold, 
            hf_token=HF_TOKEN
        )
        all_cluster_names[best_threshold] = cluster_names
        
        # Display named clusters
        display_named_clusters(cluster_names, best_threshold)
        
        # Save enhanced results
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # Add cluster names to dataframe
        best_results = clustering_results[best_threshold]
        processed_df['cluster_label'] = best_results['labels']
        processed_df['cluster_name'] = processed_df['cluster_label'].map(
            lambda x: cluster_names.get(x, {}).get('name', f'Cluster {x}')
        )
        
        # Save enhanced data
        enhanced_filename = f"usaa_complaints_named_clusters_{timestamp}.csv"
        processed_df.to_csv(enhanced_filename, index=False)
        print(f"\nEnhanced clustered data saved to: {enhanced_filename}")
        
        # Save cluster names summary
        import json
        names_filename = f"cluster_names_{timestamp}.json"
        with open(names_filename, 'w') as f:
            json.dump(all_cluster_names, f, indent=2)
        print(f"Cluster names saved to: {names_filename}")
    
    # Step 8: Visualize with names
    print("\n8. Creating enhanced visualizations...")
    visualize_clustering_results(embeddings, clustering_results)
    
    return {
        'embeddings': embeddings,
        'clustering_results': clustering_results,
        'cluster_names': all_cluster_names,
        'texts': complaint_texts,
        'processed_df': processed_df
    }

In [ ]:
# Execute enhanced clustering pipeline with LLM naming
if 'usaa_complaints_df' in locals() and not usaa_complaints_df.empty:
    print("Running enhanced clustering pipeline with LLM naming on USAA complaints...")
    enhanced_results = enhanced_clustering_with_naming(
        usaa_complaints_df, 
        max_samples=300,  # Smaller sample for faster processing
        distance_thresholds=[1.0, 1.5, 2.0, 2.5, 3.0]
    )
else:
    print("No USAA complaints data found. Running on sample data...")
    
    # Create more diverse sample data for testing
    sample_complaints = pd.DataFrame({
        'consumer_complaint_narrative': [
            'I was charged multiple overdraft fees when my account had sufficient funds',
            'Credit card company increased my interest rate without proper notification',
            'Mortgage payment was processed late even though I submitted it on time',
            'Insurance claim for car accident was denied without proper investigation',
            'ATM fees are excessive and not disclosed properly at the machine',
            'Credit report shows incorrect late payment that I never made',
            'Personal loan application rejected despite good credit score',
            'Customer service representative was rude and unhelpful with my issue',
            'Checking account was closed without notice or explanation',
            'Debit card transactions were declined even with available balance',
            'Home insurance premium increased dramatically without explanation',
            'Investment account fees were not disclosed upfront'
        ],
        'issue': [
            'Overdraft fees', 'Interest rate increase', 'Payment processing', 
            'Claim denial', 'ATM fees', 'Credit report error', 
            'Loan rejection', 'Customer service', 'Account closure',
            'Card declined', 'Premium increase', 'Fee disclosure'
        ],
        'product': [
            'Checking account', 'Credit card', 'Mortgage', 
            'Auto insurance', 'Checking account', 'Credit reporting',
            'Personal loan', 'Customer service', 'Checking account',
            'Debit card', 'Home insurance', 'Investment account'
        ]
    })
    
    print("Running enhanced clustering pipeline on sample data...")
    enhanced_results = enhanced_clustering_with_naming(
        sample_complaints, 
        max_samples=100,
        distance_thresholds=[0.5, 1.0, 1.5, 2.0]
    )

In [ ]:
# Additional analysis functions for named clusters
def analyze_named_clusters(enhanced_results):
    """
    Analyze clusters with their LLM-generated names
    """
    if not enhanced_results or 'cluster_names' not in enhanced_results:
        print("No named cluster results available")
        return
    
    cluster_names = enhanced_results['cluster_names']
    processed_df = enhanced_results['processed_df']
    
    print("\n" + "="*80)
    print("NAMED CLUSTER ANALYSIS SUMMARY")
    print("="*80)
    
    for threshold, names_dict in cluster_names.items():
        print(f"\n{'='*20} Distance Threshold: {threshold} {'='*20}")
        
        # Sort clusters by size
        sorted_clusters = sorted(names_dict.items(), key=lambda x: x[1]['size'], reverse=True)
        
        print(f"\nCluster Summary (sorted by size):")
        print("-" * 60)
        
        for cluster_id, info in sorted_clusters:
            name = info['name']
            size = info['size']
            percentage = (size / len(processed_df)) * 100
            
            print(f"🏷️  {name}")
            print(f"   Cluster ID: {cluster_id}")
            print(f"   Size: {size} complaints ({percentage:.1f}%)")
            
            # Show distribution by product if available
            if 'product' in processed_df.columns:
                cluster_df = processed_df[processed_df['cluster_label'] == cluster_id]
                top_products = cluster_df['product'].value_counts().head(3)
                print(f"   Top products: {', '.join([f'{prod} ({count})' for prod, count in top_products.items()])}")
            
            print()

def create_cluster_word_clouds(enhanced_results):
    """
    Create word clouds for each named cluster
    """
    try:
        from wordcloud import WordCloud
        import matplotlib.pyplot as plt
        
        if not enhanced_results or 'cluster_names' not in enhanced_results:
            print("No cluster results available for word clouds")
            return
        
        cluster_names = enhanced_results['cluster_names']
        texts = enhanced_results['texts']
        clustering_results = enhanced_results['clustering_results']
        
        for threshold, names_dict in cluster_names.items():
            results = clustering_results[threshold]
            labels = results['labels']
            
            n_clusters = len(names_dict)
            if n_clusters == 0:
                continue
                
            # Create subplots
            cols = min(3, n_clusters)
            rows = (n_clusters + cols - 1) // cols
            
            fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
            if n_clusters == 1:
                axes = [axes]
            elif rows == 1:
                axes = axes if isinstance(axes, (list, np.ndarray)) else [axes]
            else:
                axes = axes.flatten()
            
            fig.suptitle(f'Cluster Word Clouds (Threshold: {threshold})', fontsize=16)
            
            for i, (cluster_id, info) in enumerate(names_dict.items()):
                if i >= len(axes):
                    break
                    
                # Get texts for this cluster
                cluster_mask = labels == cluster_id
                cluster_texts = [texts[j] for j in range(len(texts)) if cluster_mask[j]]
                
                # Combine all texts in cluster
                combined_text = ' '.join(cluster_texts)
                
                # Clean text for word cloud
                cleaned_text = combined_text.replace('consumer_complaint_narrative:', '')
                cleaned_text = cleaned_text.replace('issue:', '').replace('product:', '')
                cleaned_text = cleaned_text.replace('|', ' ')
                
                if cleaned_text.strip():
                    # Generate word cloud
                    wordcloud = WordCloud(
                        width=400, height=300,
                        background_color='white',
                        max_words=50,
                        colormap='viridis'
                    ).generate(cleaned_text)
                    
                    axes[i].imshow(wordcloud, interpolation='bilinear')
                    axes[i].set_title(f"Cluster {cluster_id}: {info['name']}\n({info['size']} complaints)")
                    axes[i].axis('off')
                else:
                    axes[i].text(0.5, 0.5, 'No text available', ha='center', va='center')
                    axes[i].set_title(f"Cluster {cluster_id}: {info['name']}")
                    axes[i].axis('off')
            
            # Hide unused subplots
            for i in range(len(names_dict), len(axes)):
                axes[i].axis('off')
            
            plt.tight_layout()
            plt.show()
            
    except ImportError:
        print("WordCloud not available. Install with: pip install wordcloud")
    except Exception as e:
        print(f"Error creating word clouds: {e}")

def export_cluster_insights(enhanced_results, filename_prefix="cluster_insights"):
    """
    Export detailed cluster insights to files
    """
    if not enhanced_results:
        return
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Create insights report
    insights = []
    
    cluster_names = enhanced_results.get('cluster_names', {})
    processed_df = enhanced_results.get('processed_df', pd.DataFrame())
    
    for threshold, names_dict in cluster_names.items():
        threshold_insights = {
            'threshold': threshold,
            'total_clusters': len(names_dict),
            'clusters': []
        }
        
        for cluster_id, info in names_dict.items():
            cluster_insight = {
                'cluster_id': cluster_id,
                'name': info['name'],
                'size': info['size'],
                'percentage': (info['size'] / len(processed_df)) * 100 if len(processed_df) > 0 else 0,
                'sample_complaints': info['sample_texts']
            }
            
            # Add product distribution if available
            if not processed_df.empty and 'product' in processed_df.columns:
                cluster_df = processed_df[processed_df['cluster_label'] == cluster_id]
                product_dist = cluster_df['product'].value_counts().to_dict()
                cluster_insight['product_distribution'] = product_dist
            
            threshold_insights['clusters'].append(cluster_insight)
        
        insights.append(threshold_insights)
    
    # Save insights
    import json
    insights_file = f"{filename_prefix}_{timestamp}.json"
    with open(insights_file, 'w') as f:
        json.dump(insights, f, indent=2)
    
    print(f"Cluster insights exported to: {insights_file}")
    
    return insights

# Run additional analysis if we have enhanced results
if 'enhanced_results' in locals() and enhanced_results:
    print("\nRunning additional analysis on named clusters...")
    
    # Analyze named clusters
    analyze_named_clusters(enhanced_results)
    
    # Create word clouds
    print("\nGenerating word clouds for clusters...")
    create_cluster_word_clouds(enhanced_results)
    
    # Export insights
    print("\nExporting cluster insights...")
    insights = export_cluster_insights(enhanced_results)
    
    print("\n✅ Enhanced clustering analysis complete!")
    print("Files generated:")
    print("- Clustered data CSV with cluster names")
    print("- Cluster names JSON")
    print("- Cluster insights JSON")
    print("- Word cloud visualizations")
else:
    print("Enhanced results not available. Please run the clustering pipeline first.")